# External AF3-ReD Output Audit

**Bu notebook AF3-ReD inference çalıştırmaz.** Yalnız mevcut output'u **salt-okunur** biçimde envanterler ve denetler. `admitted` yalnız **mapping-preparation** anlamındadır; downstream kabul değildir.

Python 3.10+ ve standart kütüphane yeterlidir; CPU runtime kullanın. Kurulum, indirme, GPU, ranking, binder generation, sequence design veya geometry-core entegrasyonu yoktur. Dosya, manifest veya CIF yazılmaz/onarılamaz; sonuçlar yalnız bellekte ve hücre çıktılarındadır.

Colab'da bu notebooku açın ve aşağıdaki gömülü adapter (1–5) ile bridge (6) hücrelerini çalıştırın. `src` importu gerekmez. Sonra **output/profil → envanter → artifact ve explicit binding → readiness → kontrol** sırasını izleyin. Boş varsayılanlarla “Run all” dosya okumadan bekler. Tam envanter bellekte tutulur; ekran satır sınırı seçim veya sıralama kararı değildir.

Ana `diffusion_fixed.ipynb` akışından bağımsızdır. Üç hedef için aynı akışı her output kökü ve binding için ayrı çalıştırabilirsiniz; hedef kimliği dosyadan türetilmez.


In [ ]:
"""AF3-ReD external inventory / pre-admission only (Python 3.10+, stdlib).

Paste cells marked # %% into Colab, or import this module. No import-time I/O,
writes, subprocesses, inference, ranking, geometry or core-schema integration.
An admitted artifact is ready ONLY for explicit mapping preparation.
"""

# %% 1. Records and read-only helpers
from dataclasses import asdict, dataclass, replace
import hashlib
import json
import math
import os
from pathlib import Path
import re
import stat

ADAPTER_VERSION = "0.2"
_SAMPLE = re.compile(r"seed-(0|[1-9][0-9]*)_sample-(0|[1-9][0-9]*)")
_FORMATS = {".cif": "mmcif", ".mmcif": "mmcif", ".pdb": "pdb"}
_SIDECAR = "af3red_adapter_metadata.json"  # Assumed schema, NOT upstream output.


@dataclass(frozen=True)
class AF3ReDRunIdentity:
    run_id: str                 # Local identity: canonical path + input byte hash.
    directory: str
    job_name: str
    input_path: str
    input_sha256: str
    model_seeds: tuple
    declared_ligands: tuple     # Input declarations, not observed ligand presence.
    bias_sigma: float | None
    bias_weight: float | None   # ReD bias parameter, NOT checkpoint weights.
    declaration: dict | None
    provenance: tuple
    missing: tuple


@dataclass(frozen=True)
class StructureArtifact:
    artifact_id: str
    run_id: str
    path: str
    sha256: str
    format: str
    role: str
    seed: int | None
    sample: int | None
    status: str = "candidate"
    reasons: tuple = ()
    binding: dict | None = None
    read_provenance: dict | None = None


def _digest(data):
    return hashlib.sha256(data).hexdigest()


def _json_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")


def _path(value, root=None):
    path = Path(os.path.abspath(value))
    if path.resolve() != path:
        raise ValueError(f"Symlink aliases are unsupported: {path}")
    if root is not None and not path.is_relative_to(root):
        raise ValueError(f"Path outside declared root: {path}")
    return path


def _read(path):
    path = _path(path)
    if not stat.S_ISREG(path.stat().st_mode):
        raise ValueError(f"Not a regular file: {path}")
    before = path.stat()
    data = path.read_bytes()
    after = path.stat()
    fields = ("st_dev", "st_ino", "st_size", "st_mtime_ns", "st_ctime_ns")
    if any(getattr(before, f) != getattr(after, f) for f in fields):
        raise ValueError(f"File changed during read: {path}")
    return data, {"path": str(path), "sha256": _digest(data)}


def _unique_pairs(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key}")
        result[key] = value
    return result


def _load_json(path):
    data, provenance = _read(path)
    obj = json.loads(data, object_pairs_hook=_unique_pairs)
    _json_bytes(obj)  # Reject NaN/Infinity, including overflowing JSON numbers.
    if not isinstance(obj, dict):
        raise ValueError(f"Expected JSON object: {path}")
    return obj, provenance


def _files(root):
    root = _path(root)
    if not root.is_dir():
        raise ValueError(f"Existing output directory required: {root}")
    def fail(error):
        raise error
    for directory, dirs, files in os.walk(root, followlinks=False, onerror=fail):
        for name in sorted(dirs + files):
            if (Path(directory) / name).is_symlink():
                raise ValueError(f"Symlink in inventory root: {Path(directory) / name}")
        dirs.sort()
        for name in sorted(files):
            yield Path(directory) / name


def _text(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("Expected nonempty text")
    return value




In [ ]:
# %% 2. Run discovery: only directories anchored by exactly one *_data.json
def discover_red_runs(output_root):
    """Return AF3-compatible run candidates; layout alone cannot prove ReD use.

    Invalid/ambiguous metadata raises ValueError, without a partial return.
    No anchors means []; source/test CIFs are never treated as inference runs.
    """
    anchors = {}
    for path in _files(output_root):
        if path.name.endswith("_data.json"):
            if path.parent in anchors:
                raise ValueError(f"Ambiguous input JSONs in {path.parent}")
            anchors[path.parent] = path
    runs = []
    for directory, input_path in sorted(anchors.items()):
        if any(p != directory and directory.is_relative_to(p) for p in anchors):
            raise ValueError("Nested run directories are unsupported")
        obj, prov = _load_json(input_path)
        if obj.get("dialect") != "alphafold3" or type(obj.get("version")) is not int or obj["version"] not in (1, 2, 3):
            raise ValueError("Only alphafold3 input versions 1/2/3 are supported")
        job = re.sub(r"[^a-z0-9_.-]", "", _text(obj.get("name")).lower().replace(" ", "_"))
        if not job or input_path.name != f"{job}_data.json":
            raise ValueError("Input filename disagrees with sanitized job name")
        seeds = obj.get("modelSeeds")
        if (not isinstance(seeds, list) or not seeds
                or any(type(s) is not int or s < 0 for s in seeds)
                or len(set(seeds)) != len(seeds)):
            raise ValueError("Expected unique nonnegative integer modelSeeds")
        sequences = obj.get("sequences")
        if not isinstance(sequences, list) or not sequences:
            raise ValueError("Missing input sequences")
        ligands, ids = [], set()
        for entry in sequences:
            if not isinstance(entry, dict) or len(entry) != 1 or not set(entry) <= {"protein", "rna", "dna", "ligand"}:
                raise ValueError("Unsupported sequence entry")
            kind, entity = next(iter(entry.items()))
            if not isinstance(entity, dict):
                raise ValueError("Expected entity object")
            chain_ids = entity.get("id")
            chain_ids = [chain_ids] if isinstance(chain_ids, str) else chain_ids
            if not isinstance(chain_ids, list) or not chain_ids:
                raise ValueError("Explicit entity IDs required")
            for chain in chain_ids:
                _text(chain)
                if chain in ids:
                    raise ValueError("Duplicate input chain ID")
                ids.add(chain)
            if kind == "ligand":
                ccd, smiles = entity.get("ccdCodes"), entity.get("smiles")
                if (ccd is None) == (smiles is None):
                    raise ValueError("Ligand requires exactly one of ccdCodes/smiles")
                if ccd is not None:
                    if not isinstance(ccd, list) or not ccd:
                        raise ValueError("Expected nonempty CCD list")
                    for code in ccd:
                        _text(code)
                else:
                    _text(smiles)
                ligands.append({"ids": chain_ids, "ccdCodes": ccd, "smiles": smiles})
            else:
                _text(entity.get("sequence"))
        declaration, provenance = None, [prov]
        sigma = weight = None
        missing = ["af3red_execution_declaration_missing", "bias_sigma_unknown", "bias_weight_unknown"]
        sidecar = directory / _SIDECAR
        if sidecar.exists():
            declaration, side_prov = _load_json(sidecar)
            expected = {"schema_version", "producer", "declared_run_id", "input_sha256",
                        "bias_sigma", "bias_weight", "source_revision", "evidence_note"}
            if set(declaration) != expected or declaration["schema_version"] != "0.1" or declaration["producer"] != "AF3-ReD":
                raise ValueError("Unsupported adapter sidecar (assumed schema 0.1)")
            if declaration["input_sha256"] != prov["sha256"]:
                raise ValueError("Sidecar input hash mismatch")
            for key in ("declared_run_id", "evidence_note"):
                _text(declaration[key])
            if declaration["source_revision"] is not None:
                _text(declaration["source_revision"])
            sigma, weight = declaration["bias_sigma"], declaration["bias_weight"]
            missing = []
            for key, value, lower in (("bias_sigma", sigma, 0.1), ("bias_weight", weight, 0.0)):
                if value is None:
                    missing.append(key + "_unknown")
                elif type(value) not in (float, int) or not math.isfinite(value) or value < lower:
                    raise ValueError(f"Invalid declared {key}")
            provenance.append(side_prov)
        run_id = _digest(_json_bytes([str(directory), prov["sha256"]]))
        runs.append(AF3ReDRunIdentity(run_id, str(directory), job, str(input_path),
                    prov["sha256"], tuple(seeds), tuple(ligands), sigma, weight,
                    declaration, tuple(provenance), tuple(missing)))
    return tuple(runs)


def _verify_run(run):
    for record in run.provenance:
        if _read(record["path"])[1] != record:
            raise ValueError("Run metadata changed since discovery")
    if run.declaration is None and (Path(run.directory) / _SIDECAR).exists():
        raise ValueError("Run metadata appeared since discovery; rediscover")




In [ ]:
# %% 3. Structure inventory: no coordinate interpretation
def _inventory_artifact(path, run):
    path = _path(path, _path(run.directory))
    compressed = any(path.name.lower().endswith(s + ".gz") for s in _FORMATS)
    if path.suffix.lower() not in _FORMATS and not compressed:
        return None
    data, prov = _read(path)
    relative = path.relative_to(run.directory)
    role, seed, sample = "unrecognized_or_converted", None, None
    reasons, status = ["unrecognized_or_converted_layout"], "candidate"
    if len(relative.parts) == 1 and path.name == f"{run.job_name}_model.cif":
        role, reasons = "upstream_selected_copy", ["root_copy_not_an_independent_sample"]
    elif len(relative.parts) == 2 and (match := _SAMPLE.fullmatch(path.parent.name)):
        seed, sample = map(int, match.groups())
        if path.name == f"{run.job_name}_{path.parent.name}_model.cif":
            role, reasons = "native_sample", []
        if seed not in run.model_seeds:
            status, reasons = "rejected", ["seed_not_in_input"]
    if compressed:
        status, reasons = "rejected", ["compressed_structure_unsupported"]
    if not data:
        status, reasons = "rejected", ["empty_structure"]
    artifact_id = _digest(_json_bytes([run.run_id, str(relative), prov["sha256"]]))
    return StructureArtifact(artifact_id, run.run_id, str(path), prov["sha256"],
                             _FORMATS.get(path.suffix.lower(), "unsupported"), role, seed, sample,
                             status, tuple(reasons))


def discover_structure_files(run):
    """Inventory structures inside one run; retain root copy as a separate role.

    Only exact native sample CIF names can advance to pre-admission. Renamed,
    converted or compressed structures remain visible but cannot be promoted.
    """
    _verify_run(run)
    root, artifacts = _path(run.directory), []
    for path in _files(root):
        if path.name.endswith("_data.json") and str(path) != run.input_path:
            raise ValueError("Unexpected/nested input JSON; rediscover runs")
        artifact = _inventory_artifact(path, run)
        if artifact is not None:
            artifacts.append(artifact)
    _verify_run(run)
    return tuple(artifacts)




In [ ]:
# %% 4. Explicit, narrow readiness gate (reader supplied by the notebook)
def _check_reader_profile(profile, reader):
    if profile not in ("strict", "af3_native_v1"):
        raise ValueError("Unknown reader profile; expected strict or af3_native_v1")
    if profile == "af3_native_v1" and reader is not None:
        raise ValueError("Native profile selects its bridge explicitly; reader= is strict-only")
    if reader is not None and (not callable(reader) or getattr(reader, "reader_profile", "strict") != "strict"):
        raise ValueError("reader= requires a strict callback; select profile='af3_native_v1' for the bridge")


def _get_native_reader():
    if globals().get("__package__"):
        from .af3_native_bridge import read_af3_native_cif
    else:
        # Standalone notebook: bridge cell must be executed explicitly first.
        read_af3_native_cif = globals().get("read_af3_native_cif")
    if not callable(read_af3_native_cif) or getattr(read_af3_native_cif, "reader_profile", None) != "af3_native_v1":
        raise ValueError("AF3 native bridge unavailable; execute its notebook cell first")
    return read_af3_native_cif


def validate_basic_mapping_readiness(artifact, run, *, binding=None, reader=None, profile="strict"):
    """Return a NEW record; no profile sniffing or automatic reader fallback.

    Strict binding: artifact_sha256, target_id, model_id, auth_chain_id, segment.
    Native binding: artifact_sha256, target_id, model_id, auth_asym_id, label_asym_id.
    reader= remains strict-only. Invalid profile/callback combinations raise.
    This checks bytes and explicitly selected observed identities, not receptor
    identity, sequence correspondence, completeness or downstream admission.
    """
    _check_reader_profile(profile, reader)
    read_provenance = {"profile": profile, "read_completed": False}

    def result(status, *reasons, bound=None):
        return replace(artifact, status=status, reasons=tuple(reasons), binding=bound,
                       read_provenance=json.loads(_json_bytes(read_provenance)))
    try:
        _verify_run(run)
        path = _path(artifact.path, _path(run.directory))
        if artifact.run_id != run.run_id or _read(path)[1]["sha256"] != artifact.sha256:
            raise ValueError("Artifact/run or byte hash mismatch")
        # Never trust a caller-modified role/seed/status to authorize promotion.
        current = _inventory_artifact(path, run)
        if current is None or any(getattr(current, k) != getattr(artifact, k) for k in
                                  ("artifact_id", "format", "role", "seed", "sample")):
            raise ValueError("Artifact inventory identity mismatch")
        if current.status == "rejected":
            return result("rejected", *current.reasons)
        if artifact.role != "native_sample":
            return result("candidate", *current.reasons)
        if binding is None:
            return result("candidate", *run.missing, "explicit_target_model_chain_binding_required")
        required = {"artifact_sha256", "target_id", "model_id"} | (
            {"auth_asym_id", "label_asym_id"} if profile == "af3_native_v1" else {"auth_chain_id", "segment"})
        if not isinstance(binding, dict) or set(binding) != required:
            raise ValueError("Invalid binding fields")
        binding = json.loads(_json_bytes(binding))  # Detached audit copy.
        if binding["artifact_sha256"] != artifact.sha256:
            raise ValueError("Binding artifact hash mismatch")
        for key in ("target_id", "model_id"):
            _text(binding[key])
        if profile == "af3_native_v1":
            for key in ("auth_asym_id", "label_asym_id"):
                _text(binding[key])
            native_reader = _get_native_reader()
            read_provenance.update(reader="read_af3_native_cif", reader_version=native_reader.reader_version)
            native = native_reader(path, model_id=binding["model_id"])
            read_provenance.update(native.provenance(), read_completed=True)
            if native.source_sha256 != artifact.sha256 or native.selected_model != binding["model_id"]:
                raise ValueError("Native reader source hash/model mismatch")
            if not any(row.fields["_atom_site.group_pdb"] == "ATOM" and
                       row.fields["_atom_site.auth_asym_id"] == binding["auth_asym_id"] and
                       row.fields["_atom_site.label_asym_id"] == binding["label_asym_id"] for row in native.rows):
                raise ValueError("Selected ATOM author/label chain pair absent")
        else:
            _text(binding["auth_chain_id"])
            if type(binding["segment"]) is not int or binding["segment"] < 0:
                raise ValueError("Invalid segment")
            if reader is None:
                return result("candidate", *run.missing, "structure_reader_required", bound=binding)
            read_provenance.update(reader=getattr(reader, "__module__", "") + "." +
                                   getattr(reader, "__qualname__", type(reader).__name__))
            structure = reader(path, model_id=binding["model_id"])
            if not all(hasattr(structure, name) for name in ("source_hash", "model_id", "atoms", "warnings")):
                raise ValueError("Strict callback must return the strict Structure API; no native bridge substitution")
            read_provenance.update(source_sha256=structure.source_hash, selected_model=structure.model_id,
                                   read_completed=True)
            if structure.source_hash != artifact.sha256 or str(structure.model_id) != binding["model_id"]:
                raise ValueError("Reader source hash/model mismatch")
            if not structure.atoms or any(not all(math.isfinite(v) for v in atom.xyz) for atom in structure.atoms):
                raise ValueError("Empty/nonfinite coordinate records")
            if any(w["code"] in {"duplicate_atom_id", "duplicate_residue_id"} for w in structure.warnings):
                raise ValueError("Ambiguous atom/residue identity")
            if any(a.altloc for a in structure.atoms):
                raise ValueError("Alternate locations require a later explicit policy")
            if not any(a.group == "ATOM" and (a.chain_id, a.segment) ==
                       (binding["auth_chain_id"], binding["segment"]) for a in structure.atoms):
                raise ValueError("Selected ATOM author chain/segment absent")
        _verify_run(run)
        if _read(path)[1]["sha256"] != artifact.sha256:
            raise ValueError("Artifact changed during validation")
        if run.missing:
            return result("candidate", *run.missing, bound=binding)
        return result("admitted", "external_mapping_preparation_only", bound=binding)
    except (OSError, ValueError, UnicodeError) as exc:
        return result("rejected", f"{type(exc).__name__}: {exc}")




In [ ]:
# %% 5. In-memory manifest, deliberately separate from the core run_manifest
def build_red_manifest(output_root, *, bindings=None, reader=None, profile="strict"):
    """bindings maps artifact_id -> explicit binding; never writes a manifest.

    Unknown bindings/invalid run metadata abort the call (no partial return).
    Per-structure failures remain rejected inventory records. Rebuild to recheck.
    """
    _check_reader_profile(profile, reader)
    bindings = {} if bindings is None else json.loads(_json_bytes(bindings))
    if not isinstance(bindings, dict):
        raise ValueError("bindings must be an artifact-ID dictionary")
    runs = discover_red_runs(output_root)
    artifacts = []
    for run in runs:
        for artifact in discover_structure_files(run):
            artifacts.append(validate_basic_mapping_readiness(
                artifact, run, binding=bindings.get(artifact.artifact_id), reader=reader, profile=profile))
    if set(bindings) - {a.artifact_id for a in artifacts}:
        raise ValueError("Binding refers to an undiscovered/stale artifact ID")
    for run in runs:
        _verify_run(run)
    manifest = {"schema": "af3red_external_inventory/0.2", "adapter_version": ADAPTER_VERSION,
                "reader_profile": profile,
                "scope": "external_conformer_inventory_and_admission_prep",
                "status_scope": "admitted_means_mapping_preparation_only",
                "root": str(_path(output_root)), "runs": [asdict(r) for r in runs],
                "artifacts": [asdict(a) for a in artifacts],
                "diagnostics": (["no_recognized_runs"] if not runs else []) +
                               (["no_structure_artifacts"] if not artifacts else [])}
    manifest = json.loads(_json_bytes(manifest))
    manifest["manifest_sha256"] = _digest(_json_bytes(manifest))
    return manifest


## 6. Explicit AF3 native bridge

Ayrı native kayıt tipi; strict reader veya Structure yerine geçmez. Alanlar CIF namespace’leriyle korunur. Bu hücre dosya okumaz.


In [ ]:
"""Read-only AF3 native atom-site profile; standalone Python 3.10+ notebook cell.

Not a strict Structure reader or a CIF converter. Returns only present atom-site
fields, with explicit CIF namespaces and raw values. Does not infer author atom
names, author component names, segments, sequences or receptor correspondence.
Profile selection asserts a format contract, not proof of the producing software.
"""
from dataclasses import asdict, dataclass
import hashlib
import math
from pathlib import Path
import re
import stat

AF3_NATIVE_PROFILE = "af3_native_v1"
AF3_NATIVE_READER_VERSION = "0.1"


class AF3NativeFormatError(ValueError):
    """Unsupported/ambiguous native input; never yields a partial result."""


@dataclass(frozen=True)
class AF3NativeAtomRow:
    # Keys are actual, present CIF tags (case-normalized), never generic aliases.
    fields: dict
    source_lines: tuple
    quoted_fields: tuple


@dataclass(frozen=True)
class AF3NativeCIF:
    source_path: str
    source_sha256: str
    selected_model: str
    available_models: tuple
    columns: tuple               # Original header spelling/order.
    rows: tuple                  # Selected-model rows, including HETATM.

    def provenance(self):
        return {"profile": AF3_NATIVE_PROFILE, "reader": "read_af3_native_cif",
                "reader_version": AF3_NATIVE_READER_VERSION,
                "source_path": self.source_path, "source_sha256": self.source_sha256,
                "selected_model": self.selected_model,
                "available_models": list(self.available_models),
                "columns": list(self.columns),
                "absent_fields": ["_atom_site.auth_atom_id", "_atom_site.auth_comp_id"],
                "field_policy": "present_cif_fields_only; no_namespace_substitution",
                "producer_verified": False}

    def to_dict(self):
        return {**asdict(self), "provenance": self.provenance()}


def _native_tokens(text):
    """Small CIF 1.1 lexer, independent of the unchanged strict reader."""
    lines = text.splitlines()
    if lines and lines[0].startswith("#\\#CIF_2.0"):
        raise AF3NativeFormatError("CIF 2.0 unsupported")
    i = 0
    while i < len(lines):
        line = lines[i]
        if line.startswith(";"):
            start, parts = i + 1, [line[1:]]
            i += 1
            while i < len(lines) and not lines[i].startswith(";"):
                parts.append(lines[i])
                i += 1
            if i == len(lines) or lines[i][1:].strip():
                raise AF3NativeFormatError("Unterminated/ambiguous CIF text field")
            yield ("\n".join(parts), start, True)
        else:
            j = 0
            while j < len(line):
                if line[j].isspace():
                    j += 1
                    continue
                if line[j] == "#":
                    break
                if line[j] in "\"'":
                    quote, start = line[j], j + 1
                    j += 1
                    while j < len(line) and not (line[j] == quote and
                                                (j + 1 == len(line) or line[j+1].isspace())):
                        j += 1
                    if j == len(line):
                        raise AF3NativeFormatError(f"Unterminated quote at line {i+1}")
                    yield (line[start:j], i + 1, True)
                    j += 1
                else:
                    start = j
                    while j < len(line) and not line[j].isspace():
                        j += 1
                    value = line[start:j]
                    if value.startswith(("[", "]", "{", "}", "$")):
                        raise AF3NativeFormatError("Unsupported CIF token")
                    yield (value, i + 1, False)
        i += 1


def _native_control(token):
    value, _, quoted = token
    low = value.lower()
    return not quoted and (low.startswith(("_", "data_", "save_")) or
                           low in ("loop_", "stop_", "global_"))


def _native_table(text):
    tokens = list(_native_tokens(text))
    i, blocks, tables, seen_tags = 0, 0, [], set()
    while i < len(tokens):
        value, _, quoted = tokens[i]
        low = value.lower()
        if not quoted and low.startswith("data_"):
            blocks += 1
            if blocks != 1 or len(value) == 5:
                raise AF3NativeFormatError("Exactly one named data block required")
            i += 1
        elif blocks == 0:
            raise AF3NativeFormatError("CIF must start with a data block")
        elif not quoted and low == "loop_":
            i += 1
            columns = []
            while i < len(tokens) and not tokens[i][2] and tokens[i][0].startswith("_"):
                columns.append(tokens[i][0])
                i += 1
            keys = [c.lower() for c in columns]
            if not keys or len(set(keys)) != len(keys) or seen_tags.intersection(keys):
                raise AF3NativeFormatError("Empty/duplicate CIF headers")
            seen_tags.update(keys)
            values = []
            while i < len(tokens) and not _native_control(tokens[i]):
                values.append(tokens[i])
                i += 1
            if not values or len(values) % len(keys):
                raise AF3NativeFormatError("Empty loop or CIF row width mismatch")
            if any(k.startswith("_atom_site.") for k in keys):
                if not all(k.startswith("_atom_site.") for k in keys):
                    raise AF3NativeFormatError("Mixed atom-site category")
                tables.append((columns, keys, values))
        elif not quoted and low.startswith("_"):
            if low.startswith("_atom_site.") or low in seen_tags:
                raise AF3NativeFormatError("Scalar atom-site or duplicate tag unsupported")
            if i + 1 == len(tokens) or _native_control(tokens[i+1]):
                raise AF3NativeFormatError("Missing scalar value")
            seen_tags.add(low)
            i += 2
        else:
            raise AF3NativeFormatError(f"Unsupported CIF syntax: {value!r}")
    if len(tables) != 1:
        raise AF3NativeFormatError("Exactly one atom-site loop required")
    return tables[0]


def read_af3_native_cif(path, *, model_id):
    """Explicit model required; no default model or implicit strict-reader fallback.

    v1 requires BOTH auth_atom_id and auth_comp_id to be absent. All input rows
    are validated before selecting a model. Nontrivial altlocs, chain aliases,
    conflicting observed residue identities and duplicate atoms fail closed.
    HETATM label_seq_id may be ./?; no label residue identity is invented for it.
    """
    if not isinstance(model_id, str) or not re.fullmatch(r"[1-9][0-9]*", model_id):
        raise AF3NativeFormatError("Explicit positive integer model ID string required")
    path = Path(path).absolute()
    if path.resolve() != path or path.suffix.lower() not in (".cif", ".mmcif"):
        raise AF3NativeFormatError("Canonical uncompressed CIF path required; no symlink aliases")
    before = path.stat()
    if not stat.S_ISREG(before.st_mode):
        raise AF3NativeFormatError("Regular file required")
    data = path.read_bytes()
    columns, keys, values = _native_table(data.decode("utf-8"))
    required = ("id group_pdb type_symbol label_atom_id label_comp_id label_asym_id "
                "label_seq_id auth_asym_id auth_seq_id label_alt_id pdbx_pdb_ins_code "
                "pdbx_pdb_model_num cartn_x cartn_y cartn_z").split()
    missing = [k for k in required if "_atom_site." + k not in keys]
    if missing:
        raise AF3NativeFormatError("Missing explicit atom-site fields: " + ", ".join(missing))
    if any("_atom_site." + k in keys for k in ("auth_atom_id", "auth_comp_id")):
        raise AF3NativeFormatError("Native v1 requires absent author atom/component fields; select strict explicitly")
    rows, models, atom_ids, atom_keys = [], set(), set(), set()
    chain_forward, chain_reverse, residue_forward, residue_reverse, residues = {}, {}, {}, {}, {}

    def one_to_one(mapping, key, value, message):
        if key in mapping and mapping[key] != value:
            raise AF3NativeFormatError(message)
        mapping[key] = value

    for start in range(0, len(values), len(keys)):
        row = dict(zip(keys, values[start:start+len(keys)]))

        def get(key, nullable=False):
            value, _, quoted = row["_atom_site." + key]
            if not value or "\n" in value:
                raise AF3NativeFormatError(f"Empty/multiline identity: {key}")
            if value in (".", "?"):
                if quoted or not nullable:
                    raise AF3NativeFormatError(f"Unknown/quoted-placeholder field: {key}")
                return None
            return value

        model, group = get("pdbx_pdb_model_num"), get("group_pdb")
        if not re.fullmatch(r"[1-9][0-9]*", model) or group not in ("ATOM", "HETATM"):
            raise AF3NativeFormatError("Unsupported model ID or record group")
        models.add(model)
        atom_id = get("id")
        if atom_id in atom_ids:
            raise AF3NativeFormatError("Duplicate atom-site ID")
        atom_ids.add(atom_id)
        for axis in ("cartn_x", "cartn_y", "cartn_z"):
            try:
                finite = math.isfinite(float(get(axis)))
            except ValueError as exc:
                raise AF3NativeFormatError("Invalid coordinate token") from exc
            if not finite:
                raise AF3NativeFormatError("Nonfinite coordinate token")
        if get("label_alt_id", nullable=True) is not None:
            raise AF3NativeFormatError("Alternate locations require an explicit later policy")
        auth_chain, label_chain = get("auth_asym_id"), get("label_asym_id")
        auth_seq, label_seq = get("auth_seq_id"), get("label_seq_id", nullable=group == "HETATM")
        if not re.fullmatch(r"[+-]?[0-9]+", auth_seq):
            raise AF3NativeFormatError("Noninteger author residue ID")
        if label_seq is not None and not re.fullmatch(r"[1-9][0-9]*", label_seq):
            raise AF3NativeFormatError("Invalid label residue ID")
        ins = get("pdbx_pdb_ins_code", nullable=True)
        comp, atom = get("label_comp_id"), get("label_atom_id")
        get("type_symbol")
        one_to_one(chain_forward, (model, auth_chain), label_chain, "Author chain aliases multiple label chains")
        one_to_one(chain_reverse, (model, label_chain), auth_chain, "Label chain aliases multiple author chains")
        # Relationships already present within CIF rows, NOT sequence correspondence.
        auth_residue = (model, auth_chain, int(auth_seq), ins)
        descriptor = (group, label_chain, label_seq, comp, auth_seq)
        one_to_one(residues, auth_residue, descriptor, "Conflicting author residue identity")
        if label_seq is not None:
            label_residue = (model, label_chain, label_seq)
            one_to_one(residue_forward, auth_residue, label_residue, "Ambiguous author/label residue relation")
            one_to_one(residue_reverse, label_residue, auth_residue, "Ambiguous label/author residue relation")
        atom_key = (auth_residue, atom)
        if atom_key in atom_keys:
            raise AF3NativeFormatError("Duplicate atom identity")
        atom_keys.add(atom_key)
        if model == model_id:
            rows.append(AF3NativeAtomRow(
                {key: token[0] for key, token in row.items()},
                tuple(sorted({token[1] for token in row.values()})),
                tuple(key for key, token in row.items() if token[2])))
    if model_id not in models:
        raise AF3NativeFormatError(f"Selected model absent: {model_id}")
    after = path.stat()
    if (path.resolve() != path or any(getattr(before, k) != getattr(after, k) for k in
            ("st_dev", "st_ino", "st_size", "st_mtime_ns", "st_ctime_ns")) or path.read_bytes() != data):
        raise AF3NativeFormatError("File changed while reading")
    return AF3NativeCIF(str(path), hashlib.sha256(data).hexdigest(), model_id,
                        tuple(sorted(models, key=int)), tuple(columns), tuple(rows))


# Explicit marker to prevent using this distinct API as a strict callback.
read_af3_native_cif.reader_profile = AF3_NATIVE_PROFILE
read_af3_native_cif.reader_version = AF3_NATIVE_READER_VERSION


## 7. Output ve profil paneli

`OUTPUT_ROOT`: runtime içinde zaten erişilebilir olan mutlak output dizini (örnek: `/content/af_output`). Bu notebook Drive bağlamaz veya veri indirmez. Symlink aliasları desteklenmez. Run discovery her run için tek bir `*_data.json` gerektirir; yoksa run-manifest doğrulaması yapılamaz. Eksik sidecar otomatik oluşturulmaz.

`strict` mevcut strict reader sözleşmesini korur. `af3_native_v1` yalnız açık native bridge yolunu seçer; otomatik format tespiti/fallback veya label → author kopyalama yoktur. Profil seçmek üreticinin AF3-ReD olduğunu kanıtlamaz.


In [ ]:
#@title Output ve reader profili
OUTPUT_ROOT = "" #@param {type:"string"}
READER_PROFILE = "strict" #@param ["strict", "af3_native_v1"]
MAX_DISPLAY_ROWS = 25 #@param {type:"integer"}


### İsteğe bağlı strict reader bağlantısı

Native profil bu callback'i kullanmaz. Strict readiness için mevcut projenin **değiştirilmemiş** `read_structure` fonksiyonunu açıkça atayın. `src` yoksa aşağıdaki hücreyi olduğu gibi bırakın: strict inventory çalışır, seçili örnek `structure_reader_required` nedeniyle `candidate` kalır. Burada reader kurulumu veya alternatif strict parser bulunmaz.


In [ ]:
STRICT_READER = None
# Yalnız mevcut structure_audit paketi runtime'da erişilebilir ise açın:
# from structure_audit.structures import read_structure
# STRICT_READER = read_structure


def _audit_context():
    if READER_PROFILE not in ("strict", "af3_native_v1"):
        raise ValueError("READER_PROFILE: strict veya af3_native_v1 seçin")
    if not isinstance(OUTPUT_ROOT, str):
        raise ValueError("OUTPUT_ROOT metin olmalı")
    if not OUTPUT_ROOT:
        return None
    root = Path(OUTPUT_ROOT)
    if not root.is_absolute() or str(root.resolve()) != OUTPUT_ROOT or not root.is_dir():
        raise ValueError("OUTPUT_ROOT mevcut, mutlak ve canonical bir dizin olmalı")
    return (OUTPUT_ROOT, READER_PROFILE)


def _selection_controls():
    return (SELECTED_ARTIFACT_ID, ARTIFACT_SHA256, TARGET_ID, MODEL_ID,
            AUTH_ASYM_ID, LABEL_ASYM_ID, STRICT_AUTH_CHAIN_ID, STRICT_SEGMENT,
            STRICT_READER if READER_PROFILE == "strict" else None)


## 8. Salt-okunur envanter

Bu hücre `discover_red_runs` ve `discover_structure_files` kullanan `build_red_manifest` akışını **binding vermeden** çalıştırır. Koordinat okuma henüz yapılmaz. Her satırda artifact ID, SHA-256, rol, yol, statü ve nedenler gösterilir. Tam liste `inventory["artifacts"]`, run provenance kayıtları `inventory["runs"]` içindedir. Artifact ID yerel yol ve byte hash'ine bağlıdır; başka köke taşındığında yeniden envanterleyin.

`upstream_selected_copy` bağımsız sample olarak kabul edilmez. Eksik `*_data.json`, geçersiz metadata veya belirsiz düzen için fallback yoktur. Kök/profil değiştikten sonra bu hücreyi yeniden çalıştırın.


In [ ]:
#@title Envanteri oluştur (salt-okunur)
inventory = None
inventory_context = None
selection_result = None
selection_controls = None
context = _audit_context()
if type(MAX_DISPLAY_ROWS) is not int or MAX_DISPLAY_ROWS < 1:
    raise ValueError("MAX_DISPLAY_ROWS pozitif tam sayı olmalı")
if context is None:
    print("Bekliyor: OUTPUT_ROOT girin; henüz dosya okunmadı.")
else:
    inventory = build_red_manifest(context[0], profile=context[1])
    inventory_context = context
    counts = {s: sum(a["status"] == s for a in inventory["artifacts"])
              for s in ("candidate", "rejected", "admitted")}
    print(json.dumps({"reader_profile": inventory["reader_profile"],
                      "status_scope": inventory["status_scope"],
                      "run_count": len(inventory["runs"]),
                      "artifact_count": len(inventory["artifacts"]),
                      "status_counts": counts,
                      "diagnostics": inventory["diagnostics"],
                      "manifest_sha256": inventory["manifest_sha256"]}, indent=2))
    for row in inventory["artifacts"][:MAX_DISPLAY_ROWS]:
        print(json.dumps({k: row[k] for k in
                         ("artifact_id", "sha256", "role", "path", "status", "reasons")},
                         ensure_ascii=False))
    if len(inventory["artifacts"]) > MAX_DISPLAY_ROWS:
        print("Görüntüleme sınırı; tam liste: inventory['artifacts']")
    if not inventory["runs"]:
        print("Run bulunamadı: run-local *_data.json gerekli. Bu sonuç gerçek-output doğrulaması değildir.")


## 9. Seçili artifact ve explicit binding paneli

Artifact ID ve SHA-256'yı envanterden **kendiniz** girin. Model ve zincir alanlarını CIF incelemenize, `TARGET_ID` değerini verdiğiniz binding kararına göre açıkça girin. Hiçbiri otomatik doldurulmaz; hedef/rezidü eşleşmesi çıkarılmaz.

Native profil: `AUTH_ASYM_ID` ve `LABEL_ASYM_ID` ayrı namespace'lerdir; birbirine kopyalanmaz. Strict profil: yalnız `STRICT_AUTH_CHAIN_ID` ve `STRICT_SEGMENT` kullanılır. Segment boş varsayılanlıdır, kullanıcı açıkça girmelidir. Kullanmadığınız profil alanlarını boş bırakın. Eksik/yanlış binding girişleri işlem başlatmaz; eski sonuç temizlenir.


In [ ]:
#@title Artifact ve explicit binding (tüm ilgili alanları elle girin)
SELECTED_ARTIFACT_ID = "" #@param {type:"string"}
ARTIFACT_SHA256 = "" #@param {type:"string"}
TARGET_ID = "" #@param {type:"string"}
MODEL_ID = "" #@param {type:"string"}
AUTH_ASYM_ID = "" #@param {type:"string"}
LABEL_ASYM_ID = "" #@param {type:"string"}
STRICT_AUTH_CHAIN_ID = "" #@param {type:"string"}
STRICT_SEGMENT = "" #@param {type:"string"}


In [ ]:
#@title Seçili artifact için mapping-readiness (salt-okunur)
selection_result = None
selection_controls = None
if not SELECTED_ARTIFACT_ID:
    print("Bekliyor: envanterden SELECTED_ARTIFACT_ID ve explicit binding girin.")
else:
    context = _audit_context()
    if context is None or inventory is None or inventory_context != context:
        raise ValueError("Kök/profil için önce envanter hücresini yeniden çalıştırın")
    matches = [a for a in inventory["artifacts"] if a["artifact_id"] == SELECTED_ARTIFACT_ID]
    if len(matches) != 1:
        raise ValueError("Artifact ID envanterde tekil olarak bulunamadı; yeniden envanterleyin")
    if not re.fullmatch(r"[0-9a-f]{64}", ARTIFACT_SHA256) or ARTIFACT_SHA256 != matches[0]["sha256"]:
        raise ValueError("Explicit artifact SHA-256 envanterle uyuşmuyor")
    if any(not isinstance(v, str) or not v.strip() for v in (TARGET_ID, MODEL_ID)):
        raise ValueError("TARGET_ID ve MODEL_ID açıkça girilmeli")
    binding = {"artifact_sha256": ARTIFACT_SHA256, "target_id": TARGET_ID, "model_id": MODEL_ID}
    if READER_PROFILE == "af3_native_v1":
        if STRICT_AUTH_CHAIN_ID or STRICT_SEGMENT:
            raise ValueError("Native profil için strict binding alanlarını boş bırakın")
        if not AUTH_ASYM_ID.strip() or not LABEL_ASYM_ID.strip():
            raise ValueError("Her iki native zincir namespace'i açıkça girilmeli")
        binding.update(auth_asym_id=AUTH_ASYM_ID, label_asym_id=LABEL_ASYM_ID)
    else:
        if AUTH_ASYM_ID or LABEL_ASYM_ID:
            raise ValueError("Strict profil için native binding alanlarını boş bırakın")
        if not STRICT_AUTH_CHAIN_ID.strip() or not re.fullmatch(r"0|[1-9][0-9]*", STRICT_SEGMENT):
            raise ValueError("Strict author zinciri ve negatif olmayan tam sayı segment açıkça girilmeli")
        binding.update(auth_chain_id=STRICT_AUTH_CHAIN_ID, segment=int(STRICT_SEGMENT))
    artifact = StructureArtifact(**matches[0])
    run = AF3ReDRunIdentity(**next(r for r in inventory["runs"] if r["run_id"] == artifact.run_id))
    checked = validate_basic_mapping_readiness(
        artifact, run, binding=binding, profile=READER_PROFILE,
        reader=STRICT_READER if READER_PROFILE == "strict" else None)
    selection_result = asdict(checked)
    selection_controls = _selection_controls()
    print(json.dumps({"status_scope": inventory["status_scope"],
                      "artifact": selection_result,
                      "run_provenance": run.provenance,
                      "run_missing": run.missing}, indent=2, ensure_ascii=False))
    print("read_completed yalnız parser okumasıdır; admitted yalnız mapping-preparation anlamındadır.")
    if READER_PROFILE == "strict" and STRICT_READER is None:
        print("Strict reader bağlı değil; strict koordinat okuması yapılmadı.")


## 10. Kısa kontrol ve sınırlar

`inventory` binding öncesi manifesttir; `selection_result` yalnız son seçili artifact'in ayrı readiness kaydıdır. İkisi bellektedir, diske veya core manifestine yazılmaz. `candidate` eksik binding/metadata/reader ya da bağımsız olmayan copy anlamına gelebilir; `rejected` nedenleri sonuçta açıkça bulunur. Native reader'ın dosyayı okuyabilmesi tek başına `admitted` yapmaz.

Aşağıdaki kontrol sadece notebook kayıt sözleşmesini denetler; boş panel gerçek-output veya golden validation sayılmaz. Repo smoke-check: `python3 -B examples/smoke_af3_red_colab.py`. Sentetik ve geçici girdilerle standalone çalışma, açık profil/binding, stale seçim reddi ve dosya değişmezliğini denetler. Mevcut opt-in golden case/test bu notebook tarafından çalıştırılmaz veya değiştirilmez.

Bilerek desteklenmeyenler: inference/setup, model ağırlıkları, otomatik metadata/sidecar üretimi, CIF onarımı, parser gevşetme, label → author kopyalama, otomatik receptor/segment/residue correspondence, ranking, ligand doğrulaması, binder generation, sequence design, geometry ve core entegrasyonu.


In [ ]:
#@title Bellekteki kayıtların dar sözleşme kontrolü
if inventory is None:
    print("Bekliyor: envanter yok; gerçek output doğrulanmadı.")
else:
    assert inventory_context == _audit_context(), "Kök/profil değişti; envanteri yeniden çalıştırın"
    assert inventory["schema"] == "af3red_external_inventory/0.2"
    assert inventory["status_scope"] == "admitted_means_mapping_preparation_only"
    assert inventory["reader_profile"] == READER_PROFILE
    for row in inventory["artifacts"]:
        assert row["status"] in {"candidate", "rejected", "admitted"}
        assert row["read_provenance"]["profile"] == READER_PROFILE
    if selection_result is not None:
        assert selection_controls == _selection_controls(), "Binding/reader değişti; readiness hücresini yeniden çalıştırın"
        assert selection_result["artifact_id"] == SELECTED_ARTIFACT_ID, "Seçimi yeniden denetleyin"
        assert selection_result["status"] in {"candidate", "rejected", "admitted"}
        assert selection_result["read_provenance"]["profile"] == READER_PROFILE
        if selection_result["status"] == "admitted":
            assert selection_result["binding"] is not None
            assert selection_result["read_provenance"]["read_completed"] is True
            assert selection_result["reasons"] == ("external_mapping_preparation_only",)
    print("Bellekteki kayıt sözleşmesi kontrol edildi; bu bir golden validation değildir.")
